# 1. Exploratory analysis

What the data looks like before any modelling: how much tumour there actually is,
how the four modalities differ, and how variable patients are.

The headline fact that shapes every later decision is the **class imbalance** -
tumour is a small fraction of each brain, and the enhancing core is a small
fraction of the tumour. That is why the loss is Dice+CE rather than plain
cross-entropy, and why patch sampling is biased toward tumour.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import CACHE_DIR, MODALITIES
from src.data import cached_case_ids, load_case, split_cases

case_ids = cached_case_ids()
train_ids, val_ids, test_ids = split_cases()
print(f'cached patients: {len(case_ids)}')
print(f'split -> train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)}')

## Class balance

Fraction of voxels belonging to each label, measured over a sample of patients.

In [ ]:
sample = case_ids[:40]
counts = np.zeros(4, dtype=np.int64)
vols = []
for cid in sample:
    _, seg = load_case(cid)
    counts += np.bincount(seg.ravel(), minlength=4)
    vols.append({'case_id': cid,
                 'WT_cm3': (seg > 0).sum() / 1000,
                 'TC_cm3': np.isin(seg, [1, 3]).sum() / 1000,
                 'ET_cm3': (seg == 3).sum() / 1000})
vols = pd.DataFrame(vols)

frac = counts / counts.sum()
labels = ['0 background', '1 necrotic', '2 oedema', '3 enhancing']
for l, f in zip(labels, frac):
    print(f'{l:<15} {f*100:7.3f}%')
print(f'
tumour is {frac[1:].sum()*100:.2f}% of all voxels after brain cropping')

## Tumour volume varies by an order of magnitude

This spread is why per-patient metrics get reported with a standard deviation:
a mean Dice alone hides that small tumours are much harder.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, region in zip(axes, ['WT', 'TC', 'ET']):
    ax.hist(vols[f'{region}_cm3'], bins=20, color='#4c72b0', edgecolor='white')
    ax.set(xlabel='volume (cm3)', ylabel='patients', title=f'{region}')
    ax.grid(alpha=0.3)
fig.suptitle('Tumour volume distribution')
fig.tight_layout(); plt.show()

display(vols[['WT_cm3','TC_cm3','ET_cm3']].describe().round(1))

## Why four modalities

Each sequence makes different tissue visible. The intensity distributions below
are after per-modality z-scoring, which is what removes the scanner-to-scanner
scale differences that would otherwise dominate.

In [ ]:
from src.preprocessing import normalize_volume

img, seg = load_case(case_ids[0])
norm = normalize_volume(img, method='zscore')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for i, m in enumerate(MODALITIES):
    axes[0].hist(img[i][img[i] > 0].ravel(), bins=100, alpha=0.5, label=m, density=True)
    axes[1].hist(norm[i][norm[i] != 0].ravel(), bins=100, alpha=0.5, label=m, density=True)
axes[0].set(title='raw intensity', xlabel='intensity')
axes[1].set(title='after per-modality z-score', xlabel='z')
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## One patient, all four modalities and the mask

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

z = int((seg > 0).sum(axis=(0,1)).argmax())
cmap = ListedColormap(['#e41a1c', '#4daf4a', '#ffd92f'])

fig, axes = plt.subplots(1, 5, figsize=(19, 4.2))
for ax, i, m in zip(axes, range(4), MODALITIES):
    ax.imshow(np.rot90(img[i][:, :, z]), cmap='gray'); ax.set_title(m.upper()); ax.axis('off')
axes[4].imshow(np.rot90(img[0][:, :, z]), cmap='gray')
axes[4].imshow(np.ma.masked_equal(np.rot90(seg[:, :, z]), 0), cmap=cmap,
               vmin=1, vmax=3, alpha=0.6, interpolation='nearest')
axes[4].set_title('mask on FLAIR'); axes[4].axis('off')
axes[4].legend(handles=[Patch(color='#e41a1c', label='necrotic'),
                        Patch(color='#4daf4a', label='oedema'),
                        Patch(color='#ffd92f', label='enhancing')],
               loc='lower right', fontsize=8)
fig.tight_layout(); plt.show()